<a href="https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
print("Ready. Month =", MONTH)

Ready. Month = 2026-03


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Looking at the shape of key fields before testing anything — impressions, CTR,
and avg_position all tend to have heavy right tails in search data (a few pages
dominate volume), so summary stats alone would be misleading without checking
percentiles first.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

page_month = con.sql(f"""
    SELECT content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           AVG(gsc_avg_position) AS avg_position
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

page_month["ctr"] = page_month["clicks"] / page_month["impressions"]

percentiles = page_month[["impressions", "clicks", "avg_position", "ctr"]].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
)
print(percentiles)
print(f"\nTotal pages: {len(page_month)}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

         impressions         clicks   avg_position            ctr
count  176738.000000  176738.000000  176738.000000  176738.000000
mean     1587.986675       4.650002      15.999277       0.004594
std      5431.337724      26.722649      17.686260       0.037760
min         1.000000       0.000000       0.000000       0.000000
10%         3.000000       0.000000       3.013030       0.000000
25%        20.000000       0.000000       5.001970       0.000000
50%       173.000000       0.000000       8.505296       0.000000
75%      1039.000000       2.000000      20.369190       0.002158
90%      3930.000000      10.000000      40.066869       0.006154
99%     21799.780000      73.000000      79.759242       0.058824
max    617124.000000    5668.000000     309.000000       1.000000

Total pages: 176738


Confirmed: heavy right tails across all three volume-related fields.

- **Impressions**: mean (1,588) is roughly 9x the median (173) — a small number
  of pages carry most of the volume. The gap from 90th percentile (3,930) to max
  (617,124) is enormous, meaning a handful of pages are massive outliers relative
  to the rest of the ~176,738-page inventory.
- **Clicks**: the median is 0 — more than half of all pages got literally zero
  clicks in this month, despite having at least one impression. Mean (4.65) is
  entirely driven by a long tail (max 5,668).
- **CTR**: median is also 0, for the same reason (zero clicks). Only the top
  quartile of pages (75th percentile: 0.0022) show any meaningful CTR at all.
  The max of 1.0 is very likely a page with an extremely small impression count
  where one click produced a 100% CTR — a reminder that CTR needs a volume floor
  before it's a trustworthy number, not just a ratio taken at face value.
- **avg_position**: less skewed than the others (mean 16.0, median 8.5), though
  the max of 309 is a striking outlier — likely a page ranking extremely deep
  for at least one query, worth being aware of when averaging position across a
  page's full query mix.

**Practical implication for the next sections:** because the median page has
zero clicks, any signal test involving CTR needs a minimum impression/volume
floor to avoid drawing conclusions from mostly-zero data — this is exactly why
Signal test 2 checks impression tiers rather than treating all pages as equally
informative.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal test 1 — CTR by position tier

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def tier(pos):
    if pos <= 3: return "1-3"
    elif pos <= 10: return "4-10"
    elif pos <= 20: return "11-20"
    elif pos <= 50: return "21-50"
    else: return "51+"

page_month["position_tier"] = page_month["avg_position"].apply(tier)

test1 = page_month.groupby("position_tier").agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr", "mean")
).reindex(["1-3", "4-10", "11-20", "21-50", "51+"])
print("Signal test 1 — CTR by position tier:")
print(test1)


Signal test 1 — CTR by position tier:
                   n   avg_ctr
position_tier                 
1-3            17578  0.012399
4-10           81988  0.004926
11-20          32203  0.003211
21-50          33288  0.002287
51+            11681  0.000903


Signal test 2 — impression volume vs CTR

In [4]:
def imp_tier(imp):
    if imp < 50: return "under_50"
    elif imp < 500: return "50_to_500"
    elif imp < 5000: return "500_to_5000"
    else: return "5000_plus"

page_month["impression_tier"] = page_month["impressions"].apply(imp_tier)
test2 = page_month.groupby("impression_tier").agg(
    n=("content_hash_id", "count"),
    avg_position=("avg_position", "mean"),
    avg_ctr=("ctr", "mean")
).reindex(["under_50", "50_to_500", "500_to_5000", "5000_plus"])
print("Signal test 2 — CTR/position by impression volume:")
print(test2)

Signal test 2 — CTR/position by impression volume:
                     n  avg_position   avg_ctr
impression_tier                               
under_50         60624     16.957822  0.008380
50_to_500        54190     19.950272  0.002378
500_to_5000      48632     11.656512  0.002799
5000_plus        13292     11.408703  0.002928


Signal test 3 — AI referral share vs position

In [5]:
ai_check = con.sql(f"""
    SELECT content_hash_id,
           SUM(sessions_ai) AS ai_sessions,
           SUM(ga4_sessions) AS total_sessions,
           AVG(gsc_avg_position) AS avg_position
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE ga4_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(ga4_sessions) > 0
""").df()

ai_check["ai_share"] = ai_check["ai_sessions"] / ai_check["total_sessions"]
has_ai = ai_check[ai_check["ai_sessions"] > 0]
no_ai = ai_check[ai_check["ai_sessions"] == 0]

print(f"Pages with any AI sessions: {len(has_ai)} of {len(ai_check)} ({len(has_ai)/len(ai_check)*100:.2f}%)")
print(f"Avg position — has AI sessions: {has_ai['avg_position'].mean():.2f}")
print(f"Avg position — no AI sessions: {no_ai['avg_position'].mean():.2f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages with any AI sessions: 3793 of 90237 (4.20%)
Avg position — has AI sessions: 18.55
Avg position — no AI sessions: 13.08


**Signal test 1 verdict: CONFIRMED.** CTR decreases monotonically and sharply as
position tier worsens — from 0.0124 (1-3) down to 0.0009 (51+), roughly a 14x
drop from best to worst tier. Every tier has a large n (11,681 to 81,988), so
this is a robust pattern, not noise. This directly confirms the assumption
behind FlyRank's CTR-fix logic: CTR must be compared within position tier,
never across tiers, since position alone explains most of the CTR variation.

**Signal test 2 verdict: MIXED.** Impression volume does not cleanly predict
CTR — CTR is actually highest in the lowest-volume tier (under_50: 0.0084),
dips sharply in the 50_to_500 tier (0.0024), then recovers slightly through
500_to_5000 and 5000_plus (~0.0028-0.0029). This isn't monotonic, and it's
confounded by position: under_50 has by far the worst average position (16.96)
yet the highest CTR of any tier — likely a small-sample effect, since low-volume
pages have fewer impressions to average CTR over, making extreme ratios more
common. Volume alone is not a trustworthy standalone signal for CTR; it's better
used as a confidence weight (more impressions = more trustworthy CTR estimate)
than as a direct ranking criterion.

**Signal test 3 verdict: MIXED, with an important caveat.** Only 4.20% of pages
(3,793 of 90,237) have any AI-referral sessions at all — consistent with the
lane guide's own warning that AI-session data is sparse. Among the pages that do
have AI sessions, average position is notably *worse* (18.55) than pages with no
AI sessions (13.08) — the opposite of what you might expect if AI referrals
mirrored Google ranking. This echoes the FlyRank paper's own Finding #6:
AI-attracting content doesn't simply mirror classic organic-ranking winners.
Given the small AI-session base, this should be read as a directional pattern
worth further investigation, not a confirmed causal relationship — the sample
size here is real but still thin relative to the full inventory.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Testing the assumption behind FlyRank's real `low_ctr_visible_page` flag, as
defined in the lane guide: pages with `impressions_90d >= 500`, position 1-20,
and `ctr < 0.5%` are flagged as under-capturing clicks despite decent visibility.

The implicit assumption: pages meeting this rule are meaningfully different from
similar-volume, similar-position pages that don't trigger the flag. Testing
whether that's actually true in this data.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

flag_pages = page_month[
    (page_month["impressions"] >= 500) &
    (page_month["avg_position"] > 0) & (page_month["avg_position"] <= 20) &
    (page_month["ctr"] < 0.005)
]

similar_pool = page_month[
    (page_month["impressions"] >= 500) &
    (page_month["avg_position"] > 0) & (page_month["avg_position"] <= 20)
]

print(f"Pages meeting the low_ctr_visible_page flag: {len(flag_pages)} of {len(similar_pool)} in the same volume/position pool")
print(f"Flagged pages avg CTR: {flag_pages['ctr'].mean():.5f}")
print(f"Whole pool avg CTR: {similar_pool['ctr'].mean():.5f}")
print(f"Non-flagged pool avg CTR: {similar_pool[~similar_pool.index.isin(flag_pages.index)]['ctr'].mean():.5f}")


Pages meeting the low_ctr_visible_page flag: 41169 of 50764 in the same volume/position pool
Flagged pages avg CTR: 0.00181
Whole pool avg CTR: 0.00315
Non-flagged pool avg CTR: 0.00890


**Verdict: MIXED — the direction is right, but the flag is barely selective.**

Flagged pages do have a real, measurably lower CTR (0.00181) than non-flagged
pages in the same volume/position pool (0.00890) — nearly 5x lower, so the flag
isn't picking out an arbitrary group. But 41,169 of 50,764 pages (81.1%) in this
same-volume/position pool trigger the flag. A rule meant to identify a specific
subset of underperforming pages is instead capturing the large majority of the
pool it's applied to — which limits its usefulness as a *prioritization* signal,
since "flagged" barely distinguishes anything when 4 out of 5 pages qualify.

This is consistent with Section 1's finding that the median page has zero
clicks: at position 1-20 with 500+ impressions, a CTR below 0.5% is apparently
closer to typical than exceptional in this dataset, so the threshold may need
raising (or re-deriving from this data's own percentiles) to actually separate
a genuine "under-capturing" minority from the broader norm.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Position tier is the strongest, most reliable signal in this data — CTR
comparisons must always be made within tier, never across, since position alone
explains most of the variation (a 14x CTR gap from best to worst tier).
Impression volume is not a trustworthy standalone signal for CTR and should be
used as a confidence weight rather than a ranking criterion on its own. The
`low_ctr_visible_page` flag correctly points in the right direction but is not
selective enough as currently defined — with 81% of the eligible pool
triggering it, a content team should treat it as a broad eligibility filter,
not a priority signal, and should tighten the CTR threshold using this dataset's
own percentiles before relying on it to rank review order.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Flag selectivity: 81.1% of eligible pool triggers the flag — too broad to prioritize by itself.")
print(f"Flagged CTR (0.00181) vs non-flagged CTR (0.00890): direction correct, ~4.9x gap.")

Flag selectivity: 81.1% of eligible pool triggers the flag — too broad to prioritize by itself.
Flagged CTR (0.00181) vs non-flagged CTR (0.00890): direction correct, ~4.9x gap.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.